# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Greemines/Flyrank-Notebook-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["trend_direction"].value_counts(dropna=False))

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Basic checks
print("Rows:", len(df))
print("Unique clients:", df["client_id"].nunique())

print("\nRows per client:")
print(df.groupby("client_id").size().describe())

print("\nMissing values in modeling fields:")
print(
    df[
        ["client_id", "impressions_90d", "ctr", "avg_position", "trend_direction"]
    ].isna().sum()
)

print("\nInfinite values in numeric features:")
for col in ["impressions_90d", "ctr", "avg_position"]:
    print(col, np.isinf(df[col]).sum())

print("\nFeature ranges:")
print(
    df[["impressions_90d", "ctr", "avg_position"]].describe()
)

Rows: 30000
Unique clients: 32

Rows per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

Missing values in modeling fields:
client_id          0
impressions_90d    0
ctr                0
avg_position       0
trend_direction    0
dtype: int64

Infinite values in numeric features:
impressions_90d 0
ctr 0
avg_position 0

Feature ranges:
       impressions_90d           ctr  avg_position
count     30000.000000  30000.000000   30000.00000
mean       5200.366300      0.510733      16.34238
std       16838.019547      3.279162      15.21679
min           1.000000      0.000000       0.00000
25%          81.000000      0.000000       6.20000
50%         731.000000      0.070000      10.80000
75%        3615.250000      0.290000      22.30000
max      517715.000000    100.000000     245.00000


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

##**Method choice**

I use Logistic Regression because my task is to predict whether a page is declining. I convert trend_direction into a binary target: down means declining and all other directions mean not declining. Logistic Regression is a simple and interpretable first learned model, and its predicted probabilities can be used to rank pages for a review queue. This makes it suitable for a direct comparison with my Week-4 baseline using the same evaluation metric.

In [5]:


features = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

df_model = df[
    features + ["client_id", "trend_direction", "content_id"]
].copy()

# Binary target:
# 1 = declining
# 0 = not declining
df_model["target"] = (
    df_model["trend_direction"] == "down"
).astype(int)

print("Features:", features)
print("\nTarget counts:")
print(df_model["target"].value_counts())

print("\nTarget rate:")
print(round(df_model["target"].mean() * 100, 2), "% declining")

print("\nRows used:", len(df_model))
print("Clients used:", df_model["client_id"].nunique())


Features: ['impressions_90d', 'ctr', 'avg_position']

Target counts:
target
1    16262
0    13738
Name: count, dtype: int64

Target rate:
54.21 % declining

Rows used: 30000
Clients used: 32


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

##**Split design**

I use a grouped split by client_id so pages from the same client cannot appear in both training and test data. This is more honest than randomly splitting pages because the model should be tested on clients it did not see during training. I hold out approximately 20% of clients for the test set and use the same test rows for both the learned model and the baseline.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

X = df_model[features]
y = df_model["target"]
groups = df_model["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nClient overlap:", len(train_clients & test_clients))

print("\nTraining target rate:",
      round(train_df["target"].mean() * 100, 2), "%")

print("Test target rate:",
      round(test_df["target"].mean() * 100, 2), "%")

Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Client overlap: 0

Training target rate: 55.01 %
Test target rate: 51.1 %


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd


X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df["target"]
y_test = test_df["target"]


X_train["impressions_90d"] = np.log1p(
    X_train["impressions_90d"]
)

X_test["impressions_90d"] = np.log1p(
    X_test["impressions_90d"]
)


model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])



model.fit(X_train, y_train)


model_prob = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Model trained successfully.
Training rows: 23837
Test rows: 6163


In [9]:

results = test_df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "ctr",
        "avg_position",
        "target",
        "trend_direction"
    ]
].copy()

results["model_score"] = model_prob


model_ranking = results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

model_ranking["model_rank"] = (
    model_ranking.index + 1
)

print("Top 10 model predictions:")
display(
    model_ranking[
        [
            "model_rank",
            "content_id",
            "model_score",
            "target",
            "trend_direction"
        ]
    ].head(10)
)

Top 10 model predictions:


,model_rank,content_id,model_score,target,trend_direction
0,1,content_8c19996aa890,0.796025,1,down
1,2,content_5fe46e04994d,0.795426,1,down
2,3,content_4c36c775b818,0.791912,1,down
3,4,content_db5989a78dd3,0.783450,0,up
4,5,content_9532f197bbc8,0.778049,1,down
5,6,content_73c54f78c06a,0.771596,0,stable
6,7,content_c84a0ab98e90,0.771142,0,stable
7,8,content_cea79ef51519,0.769601,1,down
8,9,content_2db251d1a841,0.768289,0,stable
9,10,content_e12868d1f396,0.762968,0,stable


In [10]:

baseline_test = test_df.copy()

baseline_test["impression_score"] = (
    np.log1p(baseline_test["impressions_90d"])
    / np.log1p(train_df["impressions_90d"].max())
).clip(0, 1)

train_ctr_max = train_df["ctr"].max()

if train_ctr_max == 0:
    baseline_test["ctr_score"] = 0
else:
    baseline_test["ctr_score"] = (
        1 - baseline_test["ctr"] / train_ctr_max
    ).clip(0, 1)

train_position_max = train_df["avg_position"].max()

if train_position_max == 0:
    baseline_test["position_score"] = 0
else:
    baseline_test["position_score"] = (
        1 - baseline_test["avg_position"] / train_position_max
    ).clip(0, 1)


baseline_test["baseline_score"] = (
    0.50 * baseline_test["impression_score"] +
    0.30 * baseline_test["ctr_score"] +
    0.20 * baseline_test["position_score"]
)


baseline_ranking = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_ranking["baseline_rank"] = (
    baseline_ranking.index + 1
)

print("Top 10 baseline predictions:")
display(
    baseline_ranking[
        [
            "baseline_rank",
            "content_id",
            "baseline_score",
            "target",
            "trend_direction"
        ]
    ].head(10)
)


Top 10 baseline predictions:


,baseline_rank,content_id,baseline_score,target,trend_direction
0,1,content_8c19996aa890,0.996927,1,down
1,2,content_5fe46e04994d,0.996151,1,down
2,3,content_4c36c775b818,0.992700,1,down
3,4,content_db5989a78dd3,0.979593,0,up
4,5,content_9532f197bbc8,0.976212,1,down
5,6,content_73c54f78c06a,0.962325,0,stable
6,7,content_c84a0ab98e90,0.961623,0,stable
7,8,content_cea79ef51519,0.960599,1,down
8,9,content_2db251d1a841,0.958533,0,stable
9,10,content_e12868d1f396,0.950314,0,stable


In [11]:
def precision_at_k(ranking_df, score_column, k):
    top_k = ranking_df.head(k)
    return top_k["target"].mean()


comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        precision_at_k(baseline_ranking, "baseline_score", 20),
        precision_at_k(model_ranking, "model_score", 20)
    ],
    "Precision@50": [
        precision_at_k(baseline_ranking, "baseline_score", 50),
        precision_at_k(model_ranking, "model_score", 50)
    ]
})

comparison["Precision@20"] = (
    comparison["Precision@20"] * 100
).round(2)

comparison["Precision@50"] = (
    comparison["Precision@50"] * 100
).round(2)

comparison

,Method,Precision@20,Precision@50
0,Week-4 Baseline,30.0,44.0
1,Logistic Regression,35.0,44.0


##**Conclusion**

Logistic Regression slightly improves the baseline for the highest-priority 20 pages, but the improvement disappears by 50 pages.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [13]:

results["predicted"] = (results["model_score"] >= 0.5).astype(int)

false_positives = results[
    (results["target"] == 0) &
    (results["model_score"] >= 0.5)
].sort_values(
    "model_score",
    ascending=False
)

print("False positives:", len(false_positives))

display(
    false_positives[
        [
            "content_id",
            "client_id",
            "model_score",
            "target",
            "trend_direction",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ].head(10)
)

False positives: 1723


,content_id,client_id,model_score,target,trend_direction,impressions_90d,ctr,avg_position
18870,content_db5989a78dd3,client_4e07408562,0.783450,0,up,345111,0.21,5.4
22028,content_73c54f78c06a,client_f369cb89fc,0.771596,0,stable,213963,0.10,4.7
6903,content_c84a0ab98e90,client_f369cb89fc,0.771142,0,stable,223271,0.03,7.8
23460,content_2db251d1a841,client_f369cb89fc,0.768289,0,stable,198671,0.18,5.6
16736,content_e12868d1f396,client_4e07408562,0.762968,0,stable,149712,0.07,2.9
15353,content_64373b8be1b4,client_4e07408562,0.760346,0,up,153520,0.18,6.4
23446,content_1aa219431528,client_4e07408562,0.760166,0,stable,151541,0.23,5.6
11599,content_32ea6aa8091a,client_4e07408562,0.757275,0,up,145501,0.16,8.7
2346,content_11900bd7941a,client_4e07408562,0.754867,0,stable,123561,0.41,2.8
29716,content_fac19fcdfb85,client_4e07408562,0.754697,0,stable,126611,0.25,5.7


In [14]:

false_negatives = results[
    (results["target"] == 1) &
    (results["model_score"] < 0.5)
].sort_values(
    "model_score",
    ascending=True
)

print("False negatives:", len(false_negatives))

display(
    false_negatives[
        [
            "content_id",
            "client_id",
            "model_score",
            "target",
            "trend_direction",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ].head(10)
)

False negatives: 1122


,content_id,client_id,model_score,target,trend_direction,impressions_90d,ctr,avg_position
9129,content_3db3c9053d5c,client_f369cb89fc,0.182138,1,down,4,25.00,1.3
18159,content_6daf2739da2c,client_8527a891e2,0.242269,1,down,6,16.67,11.3
26543,content_7764f406228e,client_8527a891e2,0.289232,1,down,1,0.00,76.0
29079,content_5f3fbd7e5521,client_f369cb89fc,0.291294,1,down,8,12.50,1.9
27526,content_eb44131698e0,client_8527a891e2,0.311520,1,down,3,0.00,77.0
18655,content_e9bc92689a75,client_8527a891e2,0.315073,1,down,1,0.00,45.0
27213,content_bb1685e447da,client_8527a891e2,0.318740,1,down,4,0.00,77.5
29988,content_9bb9a0584cae,client_8527a891e2,0.321317,1,down,2,0.00,54.0
21442,content_584e85b1ef21,client_8527a891e2,0.321338,1,down,5,0.00,81.8
12047,content_e3bbb49f7641,client_8527a891e2,0.321941,1,down,1,0.00,37.0


In [22]:

logistic_model = model.named_steps["logistic"]

feature_cols = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logistic_model.coef_[0]
})

coef_df["absolute_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(
    coef_df[
        ["feature", "coefficient"]
    ]
)

,feature,coefficient
0,impressions_90d,0.425016
1,ctr,-0.151760
2,avg_position,-0.060495


In [23]:


top20_model = model_ranking.head(20).copy()

top20_model["error"] = np.where(
    top20_model["target"] == 1,
    "Correct",
    "False Positive"
)

print("Top-20 model results:")
display(
    top20_model[
        [
            "model_rank",
            "content_id",
            "client_id",
            "model_score",
            "trend_direction",
            "error",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ]
)

print(
    "\nFalse positives in top 20:",
    (top20_model["target"] == 0).sum()
)

print(
    "Correct declining pages in top 20:",
    (top20_model["target"] == 1).sum()
)

Top-20 model results:


,model_rank,content_id,client_id,model_score,trend_direction,error,impressions_90d,ctr,avg_position
0,1,content_8c19996aa890,client_4e07408562,0.796025,down,Correct,509252,0.15,2.5
1,2,content_5fe46e04994d,client_4e07408562,0.795426,down,Correct,517715,0.14,4.2
2,3,content_4c36c775b818,client_4e07408562,0.791912,down,Correct,463103,0.41,2.3
3,4,content_db5989a78dd3,client_4e07408562,0.783450,up,False Positive,345111,0.21,5.4
4,5,content_9532f197bbc8,client_4e07408562,0.778049,down,Correct,309192,0.87,2.0
5,6,content_73c54f78c06a,client_f369cb89fc,0.771596,stable,False Positive,213963,0.10,4.7
6,7,content_c84a0ab98e90,client_f369cb89fc,0.771142,stable,False Positive,223271,0.03,7.8
7,8,content_cea79ef51519,client_f369cb89fc,0.769601,down,Correct,208798,0.23,5.2
8,9,content_2db251d1a841,client_f369cb89fc,0.768289,stable,False Positive,198671,0.18,5.6
9,10,content_e12868d1f396,client_4e07408562,0.762968,stable,False Positive,149712,0.07,2.9



False positives in top 20: 13
Correct declining pages in top 20: 7


In [24]:
comparison_errors = model_ranking[
    [
        "content_id",
        "client_id",
        "model_score",
        "target",
        "trend_direction"
    ]
].head(20).copy()

comparison_errors["model_correct"] = (
    comparison_errors["target"] == 1
)

baseline_top20 = baseline_ranking.head(20)[
    [
        "content_id",
        "baseline_score",
        "target",
        "trend_direction"
    ]
].copy()

baseline_top20["baseline_correct"] = (
    baseline_top20["target"] == 1
)

print("Model correct in top 20:",
      comparison_errors["model_correct"].sum())

print("Baseline correct in top 20:",
      baseline_top20["baseline_correct"].sum())

Model correct in top 20: 7
Baseline correct in top 20: 6


##**Conclusion**

The model makes some false-positive predictions by ranking stable or improving pages highly, even though they are not declining. In the top-20 queue, the model identifies 7 of 20 declining pages, compared with 6 of 20 for the Week-4 baseline. The Logistic Regression coefficients show which of the three input signals the model relies on most. Because the model uses the same core signals as the baseline and only provides a small improvement at Precision@20, it should be treated as a modest ranking improvement rather than evidence of a major new predictive signal.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.